In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *


In [0]:
df = spark.read.format('parquet')\
    .option('inferSchema', 'true')\
    .option('header', 'true')\
        .load('abfss://bronze@storageaccountjan2026.dfs.core.windows.net/rawdata')

In [0]:
df.display()

In [0]:
df = df.withColumn('model_category',split(col("Model_ID"),'-')[0])\
    .withColumn('model_code',split(col("Model_ID"),'-')[1])

In [0]:
df.display()

In [0]:
df.withColumn('Units_Sold',col('Units_Sold').cast('int')).display()

In [0]:
df.printSchema()

In [0]:
df = df.withColumn('Revenue_per_item',col("Revenue")/col("Units_Sold")).display()

In [0]:
w = Window.partitionBy("BranchName","Year").orderBy(col("Year").desc())
df = df.withColumn("Total_Unit_sold",sum('Units_sold').over(w))\
    .withColumn('Revenue_per_item',col("Revenue")/col("Units_Sold"))

In [0]:
w = Window.partitionBy("year").orderBy(col("Total_Unit_sold").desc())
df =df.withColumn("Rank",dense_rank().over(w))

In [0]:
df.display()

In [0]:
df_sold = df.groupBy("Year",'BranchName').agg(sum("Units_Sold").alias("Total_Unit_sold")).sort("year","Total_Unit_sold",ascending=[1,0])
df_sold.display()

In [0]:
df.write.format('parquet')\
    .mode('append')\
        .option('path','abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales')\
            .save()

In [0]:
%sql
select * from parquet.`abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales`